# Incident Response Runbook: @solana/web3.js Supply Chain Attack

**Tactic:** Initial Access → Exfiltration
**Technique:** T1195.001 — Supply Chain Compromise: Compromise Software Dependencies
**Severity:** CRITICAL

## Overview

This runbook covers incident response for the @solana/web3.js npm supply chain attack (December 2024).
Attacker-controlled publisher credentials were used to publish backdoored versions 1.95.6 and 1.95.7.
The malicious code exfiltrated Solana private keys from any downstream Node.js process that imported
the library, impacting ~400,000 weekly downloads across wallets, dApps, and server-side signers.

## MITRE ATT&CK Mapping

| Technique | ID | Description |
|---|---|---|
| Supply Chain Compromise: Compromise Software Dependencies | **T1195.001** | Backdoored npm package published via compromised publisher token |
| Exfiltration Over C2 Channel | **T1041** | Private keys HTTP POST'd to atlantic-codfish.xyz |
| Compromise Accounts: Cloud Accounts | **T1586.003** | npm publish token compromised (likely stolen from CI secrets) |

## Lateral Movement Analysis

The attack used npm's auto-update mechanism as a lateral movement vector:

1. **Publisher token theft** — Attacker obtains `@solana/web3.js` npm publish credentials
2. **Backdoor injection** — Versions 1.95.6 & 1.95.7 contain key-exfiltration hook in `Connection` constructor
3. **Automatic propagation** — CI/CD pipelines and `npm install` silently pull the malicious version across all downstream projects
4. **Developer runtime as pivot** — malicious code runs inside the same Node.js process holding Solana `Keypair` objects
5. **Wallet drain** — any `secretKey` in memory is serialized and POST'd to attacker C2

**Full lateral movement chain:**
`stolen npm token → backdoored package → downstream CI/CD → developer/server runtime → private key exfiltration → user wallet drain`

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm which package versions are affected
- Identify downstream applications that pulled 1.95.6 or 1.95.7
- Detect active exfiltration traffic to attacker C2
- Assess key compromise scope


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

print("=" * 60)
print("STEP 1: Detection & Analysis — @solana/web3.js Supply Chain")
print("=" * 60)

detection_time = datetime.now().isoformat()
affected_systems = []
splunk_indicators = []
unique_users = set()
source_hosts = set()

# Query Splunk for outbound HTTP to known C2 domains
print("\n[QUERY] Searching for outbound traffic to supply chain C2 endpoints...")
splunk_query = '''
index=network OR index=proxy
(dest_host="atlantic-codfish.xyz" OR dest_host="*solana-cdn*" OR url="*secretKey*")
| stats count by src_ip, dest_host, dest_port, uri_path, _time
| sort -count
'''
try:
    splunk_results = splunk.search_events(splunk_query, timeframe="-72h")
    print(f"   Found {len(splunk_results)} outbound C2 connection events")
except Exception as e:
    print(f"   Splunk query failed: {e}")
    splunk_results = []

for event in splunk_results:
    system_info = {
        'hostname': event.get('src_ip', 'unknown'),
        'dest': event.get('dest_host', 'unknown'),
        'uri': event.get('uri_path', ''),
        'count': event.get('count', 0),
        'last_seen': event.get('_time', detection_time)
    }
    affected_systems.append(system_info)
    source_hosts.add(event.get('src_ip', 'unknown'))
    splunk_indicators.append({
        'type': 'c2_connection',
        'value': f"{event.get('src_ip')} -> {event.get('dest_host')}{event.get('uri_path','')}",
        'context': 'Potential private key exfiltration to supply chain attacker'
    })

# Check npm audit logs / CI pipeline logs for affected versions
print("\n[QUERY] Searching CI/build system logs for malicious package versions...")
npm_query = '''
index=ci_logs OR index=build_logs
("@solana/web3.js@1.95.6" OR "@solana/web3.js@1.95.7" OR "web3.js 1.95.6" OR "web3.js 1.95.7")
| stats count by host, _time
'''
try:
    npm_results = splunk.search_events(npm_query, timeframe="-168h")
    print(f"   Found {len(npm_results)} build systems that installed malicious version")
    for r in npm_results:
        affected_systems.append({
            'hostname': r.get('host', 'unknown'),
            'reason': 'installed @solana/web3.js 1.95.6 or 1.95.7',
            'last_seen': r.get('_time', detection_time)
        })
        source_hosts.add(r.get('host', 'unknown'))
except Exception as e:
    print(f"   CI log query failed: {e}")

# Enrich with MISP threat intelligence
print("\n[ENRICHMENT] Checking MISP for supply chain IOCs...")
misp_results = []
c2_iocs = ["atlantic-codfish.xyz", "162.159.200.1"]
try:
    for ioc in c2_iocs:
        hits = misp.search_iocs(ioc)
        if hits:
            misp_results.extend(hits)
            print(f"   MISP hit for {ioc}: {len(hits)} events")
except Exception as e:
    print(f"   MISP enrichment failed: {e}")

# Create IRIS case
print("\n[CASE] Creating IRIS incident case...")
try:
    incident_data = {
        'title': f'Supply Chain Attack — @solana/web3.js 1.95.6/1.95.7 — {len(affected_systems)} systems',
        'description': 'Backdoored npm package exfiltrating Solana private keys to attacker C2',
        'severity': 'CRITICAL',
        'tactic': 'Initial Access / Exfiltration',
        'technique': 'T1195.001 Supply Chain Compromise',
        'indicators': splunk_indicators,
        'affected_systems': affected_systems,
        'threat_intelligence': misp_results
    }
    incident_id = iris.create_case(incident_data)
    print(f"   Created IRIS case: {incident_id}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")
    incident_id = f"LOCAL-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"\n✅ Detection complete:")
print(f"   - Affected systems/pipelines: {len(affected_systems)}")
print(f"   - C2 connection events: {len([i for i in splunk_indicators if i['type']=='c2_connection'])}")
print(f"   - MISP threat intel hits: {len(misp_results)}")
print(f"   - Incident ID: {incident_id}")


## Phase 2: Containment

### Objectives
- Pin `@solana/web3.js` to last known-good version (1.95.5) across all projects
- Block outbound connections to attacker C2 domains
- Rotate all potentially exposed Solana private keys
- Halt CI/CD pipelines pending audit


In [ ]:
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
containment_actions = []
blocked_ips = []
isolated_hosts = []
disabled_accounts = []

# 1. Block C2 domains and IPs at network perimeter
print("\n[CONTAINMENT] Blocking attacker C2 endpoints...")
c2_domains = ["atlantic-codfish.xyz", "solana-cdn.com"]
c2_ips = ["162.159.200.1", "104.18.32.7"]
try:
    for domain in c2_domains:
        result = shuffle.block_domain(domain)
        if result:
            containment_actions.append({'action': 'domain_block', 'target': domain, 'status': 'success', 'timestamp': containment_time})
            print(f"   Blocked domain: {domain}")
    for ip in c2_ips:
        result = shuffle.block_ip(ip)
        if result:
            blocked_ips.append(ip)
            containment_actions.append({'action': 'ip_block', 'target': ip, 'status': 'success', 'timestamp': containment_time})
            print(f"   Blocked IP: {ip}")
except Exception as e:
    print(f"   Error blocking C2 endpoints: {e}")

# 2. Isolate affected build/CI systems
print("\n[CONTAINMENT] Isolating affected CI/CD systems...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            result = crowdstrike.isolate_host(system['device_id'])
            if result:
                isolated_hosts.append(system['hostname'])
                containment_actions.append({'action': 'host_isolation', 'target': system['hostname'], 'method': 'CrowdStrike', 'status': 'success', 'timestamp': containment_time})
                print(f"   Isolated: {system['hostname']}")
except Exception as e:
    print(f"   Error isolating systems: {e}")

# 3. Force npm dependency pin across repositories
print("\n[CONTAINMENT] Pinning @solana/web3.js to safe version 1.95.5...")
pin_command = '''
# Run on all affected repositories:
# npm install @solana/web3.js@1.95.5 --save-exact
# OR add to package.json resolutions:
# "resolutions": { "@solana/web3.js": "1.95.5" }
# Commit lock files immediately.
'''
print(f"   Dependency pin script prepared (execute per repo):{pin_command}")
containment_actions.append({'action': 'npm_pin', 'target': '@solana/web3.js@1.95.5', 'status': 'manual_required', 'timestamp': containment_time})

# 4. Revoke npm publish tokens
print("\n[CONTAINMENT] Revoking compromised npm publish tokens...")
try:
    revoke_result = shuffle.revoke_api_token(service='npm', scope='publish', package='@solana/web3.js')
    if revoke_result:
        containment_actions.append({'action': 'npm_token_revoke', 'target': '@solana/web3.js npm publish token', 'status': 'success', 'timestamp': containment_time})
        print("   npm publish token revoked")
except Exception as e:
    print(f"   npm token revocation failed: {e}")

# 5. Enable enhanced monitoring for key exfiltration patterns
print("\n[CONTAINMENT] Enabling enhanced exfiltration monitoring...")
try:
    monitoring_rules = [{
        'name': 'Solana Key Exfiltration Detection',
        'query': 'index=proxy dest_host="*.xyz" OR dest_host="*solana*cdn*" http_method=POST | where len(request_body) > 32',
        'alert_threshold': 1,
        'time_window': '1m'
    }]
    splunk.enable_enhanced_monitoring(monitoring_rules)
    print("   Enhanced exfiltration monitoring enabled")
except Exception as e:
    print(f"   Monitoring setup failed: {e}")

print(f"\n✅ Containment complete:")
print(f"   - C2 domains blocked: {len(c2_domains)}")
print(f"   - C2 IPs blocked: {len(blocked_ips)}")
print(f"   - Systems isolated: {len(isolated_hosts)}")
print(f"   - npm token revoked: ✓")


## Phase 3: Eradication

### Objectives
- Remove malicious package versions from all environments
- Purge npm cache on all affected systems
- Audit all downstream packages for similar backdoors
- Rotate all Solana keypairs that ran on affected systems


In [ ]:
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

eradication_time = datetime.now().isoformat()
eradication_actions = []
rotated_keys = []
cleaned_systems = []

# 1. Remove malicious package and clear npm cache
print("\n[ERADICATION] Removing malicious npm package versions...")
eradication_script = '''
#!/bin/bash
# Run on all affected Node.js systems:
npm uninstall @solana/web3.js
npm cache clean --force
npm install @solana/web3.js@1.95.5 --save-exact
# Verify installed version:
npm list @solana/web3.js
# Verify package integrity:
npm audit
shasum -a 256 node_modules/@solana/web3.js/lib/index.cjs.js
# Expected hash for 1.95.5: (verify against npmjs.com)
'''
try:
    for system in affected_systems:
        if system.get('device_id'):
            result = crowdstrike.run_script(system['device_id'], eradication_script)
            if result:
                cleaned_systems.append(system['hostname'])
                eradication_actions.append({'action': 'npm_purge', 'target': system['hostname'], 'status': 'success', 'timestamp': eradication_time})
                print(f"   Cleaned npm cache on: {system['hostname']}")
except Exception as e:
    print(f"   Error running eradication script: {e}")

# 2. Rotate all Solana keypairs on affected systems
print("\n[ERADICATION] Rotating exposed Solana private keys...")
try:
    keypair_rotation_query = '''
    index=app_logs ("new Keypair" OR "Keypair.fromSecretKey" OR "bs58.decode")
    | stats count by host, process_name
    '''
    keypair_results = splunk.search_events(keypair_rotation_query, timeframe="-168h")
    for result in keypair_results:
        host = result.get('host', 'unknown')
        rotated_keys.append(host)
        eradication_actions.append({'action': 'keypair_rotation_flagged', 'target': host, 'status': 'manual_rotation_required', 'timestamp': eradication_time})
        print(f"   Keypair rotation required on: {host}")
    print(f"   Total systems requiring key rotation: {len(rotated_keys)}")
except Exception as e:
    print(f"   Keypair audit failed: {e}")

# 3. Audit all transitive npm dependencies for similar patterns
print("\n[ERADICATION] Auditing transitive npm dependencies for backdoor patterns...")
audit_query = '''
index=siem sourcetype=npm_audit
(severity=high OR severity=critical) package!="@solana/web3.js"
| stats count by package, vulnerability, affected_version
| sort -count
'''
try:
    audit_results = splunk.search_events(audit_query, timeframe="-24h")
    print(f"   Found {len(audit_results)} other high/critical npm vulnerabilities for review")
except Exception as e:
    print(f"   Dependency audit failed: {e}")

# 4. Verify eradication
print("\n[ERADICATION] Verifying eradication...")
try:
    verify_query = '''
    index=network dest_host="atlantic-codfish.xyz"
    | stats count by src_ip, _time
    '''
    verify_results = splunk.search_events(verify_query, timeframe="-1h")
    if len(verify_results) == 0:
        print("   ✅ No new C2 traffic detected — eradication appears complete")
    else:
        print(f"   ⚠️  Still detecting {len(verify_results)} C2 connection events — investigation ongoing")
except Exception as e:
    print(f"   Verification query failed: {e}")

print(f"\n✅ Eradication complete:")
print(f"   - Systems cleaned: {len(cleaned_systems)}")
print(f"   - Keypair rotations required: {len(rotated_keys)}")
print(f"   - Dependency audit completed: ✓")


## Phase 4: Recovery

### Objectives
- Restore CI/CD pipelines with pinned safe dependency versions
- Issue new Solana keypairs for all affected services
- Validate no residual C2 traffic
- Re-enable isolated systems


In [ ]:
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

recovery_time = datetime.now().isoformat()
recovery_actions = []
reenabled_hosts = []
restored_services = []

# 1. Re-enable isolated systems after verification
print("\n[RECOVERY] Re-enabling isolated systems...")
try:
    for host in isolated_hosts:
        system = next((s for s in affected_systems if s.get('hostname') == host), None)
        if system and system.get('device_id'):
            result = crowdstrike.reenable_host(system['device_id'])
            if result:
                reenabled_hosts.append(host)
                recovery_actions.append({'action': 'host_reenable', 'target': host, 'status': 'success', 'timestamp': recovery_time})
                print(f"   Re-enabled: {host}")
except Exception as e:
    print(f"   Error re-enabling systems: {e}")

# 2. Update package.json lock files in all repositories
print("\n[RECOVERY] Validating package lock files across repositories...")
lockfile_check = '''
#!/bin/bash
# Validate that no project references 1.95.6 or 1.95.7
grep -r "web3.js.*1\.95\.[67]" . --include="package*.json" --include="yarn.lock"
if [ $? -eq 0 ]; then
  echo "WARNING: Malicious version reference found — do not deploy"
  exit 1
else
  echo "OK: No malicious version references found"
fi
'''
recovery_actions.append({'action': 'lockfile_validation', 'target': 'all repositories', 'status': 'script_prepared', 'timestamp': recovery_time})
print("   Lock file validation script prepared")

# 3. Re-fund wallets and update signing infrastructure
print("\n[RECOVERY] Updating signing infrastructure with new keypairs...")
try:
    for host in rotated_keys:
        recovery_actions.append({'action': 'new_keypair_deployed', 'target': host, 'status': 'manual_required', 'timestamp': recovery_time})
    print(f"   {len(rotated_keys)} systems require new keypair deployment")
    print("   Action: generate new Solana keypairs, update environment variables, re-fund from cold storage")
    restored_services = list(set(rotated_keys))
except Exception as e:
    print(f"   Keypair recovery failed: {e}")

# 4. Restore monitoring to normal operations
print("\n[RECOVERY] Restoring normal monitoring...")
try:
    splunk.restore_normal_monitoring()
    crowdstrike.restore_normal_operations()
    recovery_actions.append({'action': 'monitoring_restore', 'target': 'all', 'status': 'success', 'timestamp': recovery_time})
    print("   Monitoring restored to normal operations")
except Exception as e:
    print(f"   Monitoring restore failed: {e}")

# 5. Validate recovery — confirm no active C2 for 24h
print("\n[RECOVERY] Final validation — 24h clean window check...")
try:
    final_check = splunk.search_events(
        'index=network dest_host="atlantic-codfish.xyz" | stats count',
        timeframe="-24h"
    )
    if not final_check or int(final_check[0].get('count', 0)) == 0:
        print("   ✅ 24h clean window confirmed — no C2 traffic")
    else:
        print(f"   ⚠️  C2 traffic still detected: {final_check}")
except Exception as e:
    print(f"   Final validation failed: {e}")

print(f"\n✅ Recovery complete:")
print(f"   - Hosts re-enabled: {len(reenabled_hosts)}")
print(f"   - Services requiring new keypairs: {len(restored_services)}")
print(f"   - Monitoring restored: ✓")


## Phase 5: Post-Incident Activities

### Objectives
- Document full timeline and blast radius
- Implement npm token MFA and package provenance enforcement
- Share IOCs with community (npm security team, Solana Foundation)
- Review all CI/CD secrets hygiene


In [ ]:
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_actions = []
closure_time = datetime.now().isoformat()

# 1. Generate incident report
print("\n[POST-INCIDENT] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'title': '@solana/web3.js Supply Chain Attack — IR Report',
        'detection_time': detection_time,
        'closure_time': closure_time,
        'severity': 'CRITICAL',
        'technique': 'T1195.001',
        'summary': {
            'affected_systems': len(affected_systems),
            'c2_connections_detected': len([i for i in splunk_indicators if i['type']=='c2_connection']),
            'systems_isolated': len(isolated_hosts),
            'keypairs_rotated': len(rotated_keys),
            'c2_domains_blocked': 2
        },
        'timeline': {
            'detection': detection_time,
            'containment': containment_time,
            'eradication': eradication_time,
            'recovery': recovery_time,
            'closure': closure_time
        },
        'recommendations': [
            'Enable npm 2FA/granular access tokens with publish scope restrictions',
            'Implement Sigstore/npm provenance attestation for all published packages',
            'Pin exact versions (no ^/~ ranges) for all cryptographic dependencies',
            'Add package integrity checks (shasum) to CI pipelines',
            'Never use auto-update (latest) for wallet-critical npm packages',
            'Monitor npm security advisories for all Solana ecosystem packages'
        ]
    }
    report_filename = f"solana_web3js_supply_chain_report_{incident_id}.json"
    with open(report_filename, 'w') as f:
        json.dump(incident_report, f, indent=2, default=str)
    print(f"   Report written: {report_filename}")
    post_incident_actions.append({'action': 'report_generation', 'target': report_filename, 'status': 'success', 'timestamp': closure_time})
except Exception as e:
    print(f"   Report generation failed: {e}")

# 2. Share IOCs with npm security team and Solana Foundation
print("\n[POST-INCIDENT] Sharing IOCs with community...")
try:
    iocs_to_share = [
        {'type': 'domain', 'value': 'atlantic-codfish.xyz', 'context': 'C2 for @solana/web3.js supply chain attack'},
        {'type': 'npm_package', 'value': '@solana/web3.js@1.95.6', 'context': 'Backdoored npm version'},
        {'type': 'npm_package', 'value': '@solana/web3.js@1.95.7', 'context': 'Backdoored npm version'},
    ]
    for ioc in iocs_to_share:
        misp.share_indicator(ioc, incident_id)
        post_incident_actions.append({'action': 'ioc_shared', 'target': ioc['value'], 'status': 'success', 'timestamp': closure_time})
        print(f"   Shared IOC: {ioc['value']}")
except Exception as e:
    print(f"   IOC sharing failed: {e}")

# 3. Implement preventive controls
print("\n[POST-INCIDENT] Implementing preventive controls...")
try:
    splunk.update_correlation_rules([{
        'name': 'npm Supply Chain C2 Detection',
        'search': 'index=proxy http_method=POST (dest_host="*.xyz" OR dest_host="*cdn*") request_body_length > 32 | lookup solana_keypair_indicators | where isnotnull(match)',
        'alert_threshold': 1, 'time_window': '5m'
    }])
    print("   Updated Splunk rules for npm supply chain C2 patterns")
    post_incident_actions.append({'action': 'splunk_rule_update', 'status': 'success', 'timestamp': closure_time})
except Exception as e:
    print(f"   Preventive controls update failed: {e}")

# 4. Close IRIS case
print("\n[POST-INCIDENT] Closing incident case...")
try:
    iris.close_case(incident_id, {'status': 'closed', 'closure_time': closure_time, 'resolution': 'Supply chain package patched, keys rotated, C2 blocked'})
    print(f"   IRIS case closed: {incident_id}")
except Exception as e:
    print(f"   Case closure failed: {e}")

print(f"\n✅ Post-incident activities complete:")
print(f"   - Report generated ✓")
print(f"   - IOCs shared: {len([a for a in post_incident_actions if a.get('action')=='ioc_shared'])}")
print(f"   - Detection rules updated ✓")
print(f"   - Case closed: {incident_id}")
print(f"\n🔒 @solana/web3.js Supply Chain IR Complete")
print(f"   Duration: {(datetime.fromisoformat(closure_time) - datetime.fromisoformat(detection_time)).total_seconds() / 3600:.1f} hours")


## Summary

This runbook guided response to the @solana/web3.js supply chain attack where backdoored npm
package versions exfiltrated private keys from downstream applications.

### Key Takeaways
- Supply chain attacks bypass traditional perimeter defenses — focus on dependency integrity checks
- Solana private keys in Node.js process memory are reachable by any imported module
- Pin cryptographic library versions exactly; never allow auto-update ranges (^, ~) for wallet-critical deps
- npm publisher accounts must use granular tokens with 2FA and package-scoped publish permissions
- Monitor for unexpected outbound HTTP POST requests from Node.js application servers

### Preventive Architecture
- Use hardware wallets or dedicated HSM for production signing keys
- Implement Sigstore provenance attestation for npm packages
- Add `npm audit` and shasum integrity checks to every CI/CD pipeline stage


## References

- https://github.com/solana-labs/solana-web3.js/releases — Official @solana/web3.js releases
- https://socket.dev/blog/supply-chain-attack-solana-web3-js — Socket.dev supply chain analysis
- https://attack.mitre.org/techniques/T1195/001/ — MITRE T1195.001: Supply Chain Compromise
- https://attack.mitre.org/techniques/T1041/ — MITRE T1041: Exfiltration Over C2 Channel
- https://docs.npmjs.com/threats-and-mitigations — npm security best practices
- https://docs.sigstore.dev/ — Sigstore package provenance attestation
- https://github.com/nicolo-ribaudo/tc39-proposal-source-phase-imports — TC39 module integrity proposals
